# Module 2 — Scope & Namespaces · Practice Solutions
> **Teacher's solution guide.** Each question shows the **concept** it tests, a short **how-to-explain** note, the **worked solution** (run it live), and a **common mistake** to highlight. Run every code cell with `Shift+Enter`.

### Q1. Predict the output of `x=1; def f(): x=2; f(); print(x)`
**Answer: it prints `1`.**

**Concept (LEGB):** assigning `x = 2` inside `f` creates a **local** `x`, separate from the global. Changing the local does not touch the global, so after `f()` the global `x` is still `1`.

**Explain to students:** assignment creates/affects a *local* name unless you say otherwise (`global`).

In [1]:
x = 1
def f():
    x = 2                 # local to f — a different x
    print('inside f:', x)
f()
print('outside  :', x)    # still 1

inside f: 2
outside  : 1


### Q2. Fix an `UnboundLocalError` counter — two ways
**Concept:** assigning to `count` anywhere in the function makes it local for the *whole* function, so reading it first fails.

**Explain to students:** Fix A uses `global` to rebind the module variable. Fix B avoids global state by passing the value in and returning the new one — the cleaner, testable approach.

In [2]:
# The bug:
count = 0
def increment_bad():
    count = count + 1     # count is local -> read before assignment
    return count
try:
    increment_bad()
except UnboundLocalError as e:
    print('Bug:', e)

# Fix A — global
count = 0
def increment_global():
    global count
    count += 1
    return count
increment_global(); print('Fix A:', increment_global())

# Fix B — pass in, return out (preferred)
def increment_pure(c):
    return c + 1
c = 0
c = increment_pure(c); c = increment_pure(c)
print('Fix B:', c)

Bug: cannot access local variable 'count' where it is not associated with a value
Fix A: 2
Fix B: 2


### Q3. `make_multiplier(k)` returns a function that multiplies by `k`
**Concept:** a **closure** — the inner function captures `k` from the enclosing scope and remembers it after the outer function returns.

**Explain to students:** `make_multiplier(3)` bakes `k=3` into the returned function; each returned function carries its own `k`.

In [3]:
def make_multiplier(k):
    def mult(x):
        return x * k        # k captured from the enclosing scope
    return mult

triple = make_multiplier(3)
double = make_multiplier(2)
print('triple(10):', triple(10))   # 30
print('double(10):', double(10))   # 20

triple(10): 30
double(10): 20


### Q4. Why is `def add(x, items=[])` buggy? Rewrite it.
**Concept:** a default argument is evaluated **once**, at definition time. The same list is then shared across every call, so it accumulates unexpectedly.

**Explain to students:** use `None` as the default and create a fresh list inside the function.

In [4]:
# Buggy: the default list is shared
def add_bad(x, items=[]):
    items.append(x); return items
print('bug:', add_bad(1), add_bad(2))   # [1] then [1, 2] — leaked!

# Correct: None sentinel
def add_ok(x, items=None):
    if items is None:
        items = []
    items.append(x); return items
print('ok :', add_ok(1), add_ok(2))     # [1] then [2]

bug: [1, 2] [1, 2]
ok : [1] [2]


### Q5. Use `globals()` and `locals()` to show visible names
**Concept:** every function call has its own **local** namespace; the module has a **global** one.

**Explain to students:** `locals()` shows what the function can see locally; `globals()` shows module-level names — proof that scopes are separate.

In [5]:
g = 10
def show():
    local_only = 5
    print('locals() inside show:', locals())
    print("'g' visible via globals()?", 'g' in globals())
    print("'local_only' in globals()?", 'local_only' in globals())
show()
print('module-level g =', g)

locals() inside show: {'local_only': 5}
'g' visible via globals()? True
'local_only' in globals()? False
module-level g = 10


### Q6. Counter with `nonlocal` that can also `reset()`
**Concept:** two inner functions share one enclosing variable `n` (a closure); `nonlocal` lets them **rebind** it.

**Explain to students:** returning two functions gives a tiny object with private state — `inc` and `reset` both act on the same `n`.

In [6]:
def make_counter():
    n = 0
    def inc():
        nonlocal n; n += 1; return n
    def reset():
        nonlocal n; n = 0
    return inc, reset

inc, reset = make_counter()
print(inc(), inc(), inc())   # 1 2 3
reset()
print('after reset:', inc())  # 1

1 2 3
after reset: 1
